# Question 4: Sentiment Analysis Classification

## Approach Motivation

This notebook implements a **hybrid sentiment analysis pipeline** combining two complementary paradigms:

1. **SenticNet (Knowledge-Based)** — A commonsense knowledge base that assigns polarity values and semantic primitives to ~100,000 natural language concepts. Used here for **subjectivity detection**: if a post contains at least one SenticNet-recognised concept, it is treated as opinionated content.

2. **`cardiffnlp/twitter-roberta-base-sentiment-analysis` (Machine-Learning / Transformer)** — A RoBERTa model fine-tuned on ~58M tweets for three-way sentiment (NEG / NEU / POS). Twitter and Reddit share similar informal, short-form writing styles—making this the most domain-appropriate publicly available model for our data. Used here for **polarity detection** on subjective content.

### Why Hybrid?
- Pure knowledge-based approaches suffer from recall issues (out-of-vocabulary terms, slang, abbreviations common on Reddit).
- Pure ML approaches treat all text as opinionated; they lack a principled mechanism for filtering neutral/objective posts before polarity scoring.
- Combining **SenticNet subjectivity gating → HuggingFace polarity scoring** gives us both interpretability and deep-learning accuracy.

### Pipeline
```
Text → Preprocessing → SenticNet subjectivity
                           ├─ Objective ──────────────► Label: Neutral (skip polarity)
                           └─ Subjective → HuggingFace polarity
                                               ├─ POS ► Label: Positive
                                               └─ NEG ► Label: Negative
```

In [2]:
# Install required packages
import subprocess, sys

packages = ["senticnet", "transformers", "torch", "pandas", "openpyxl", "scikit-learn", "numpy"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed successfully.")

All packages installed successfully.


In [ ]:
import pandas as pd
import numpy as np
import json
import re
import time
import random
import warnings
from pathlib import Path

from senticnet.senticnet import SenticNet
from transformers import pipeline as hf_pipeline
from sklearn.metrics import (
    classification_report, precision_recall_fscore_support, accuracy_score
)

warnings.filterwarnings("ignore")

# -- SenticNet --
sn = SenticNet()

# -- HuggingFace model --
# Try the updated CardiffNLP id first, then a BERTweet fallback.
HF_MODELS = [
    "cardiffnlp/twitter-roberta-base-sentiment-analysis-latest",
    "finiteautomata/bertweet-base-sentiment-analysis",
]

hf_sentiment = None
for model_id in HF_MODELS:
    try:
        hf_sentiment = hf_pipeline(
            "sentiment-analysis",
            model=model_id,
            tokenizer=model_id,
            truncation=True,
            max_length=512,
        )
        print(f"Loaded HuggingFace model: {model_id}")
        break
    except Exception as e:
        print(f"Skipping {model_id}: {e}")

if hf_sentiment is None:
    raise RuntimeError(
        "Could not load any sentiment model from HuggingFace. "
        "Check internet access or run `huggingface-cli login` if needed."
    )

print("SenticNet and HuggingFace model loaded successfully.")

Skipping cardiffnlp/twitter-roberta-base-sentiment-analysis-latest: cardiffnlp/twitter-roberta-base-sentiment-analysis-latest is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: finiteautomata/bertweet-base-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loaded HuggingFace model: finiteautomata/bertweet-base-sentiment-analysis
SenticNet and HuggingFace model loaded successfully.


## Data Preprocessing

We only analyze data from each topic's `enriched_results.json` file.

Text units analyzed separately:
- `post_title`
- `post_body`
- `comment_body` (all nested replies, recursively)

Cleaning steps:
- Decode HTML entities (for example `&amp;` -> `&`)
- Normalize unicode quotes/apostrophes
- Remove URLs and Reddit mentions (`u/`, `r/`)
- Remove markdown emphasis and quote prefixes
- Keep basic punctuation (`! ? . , -`) for sentiment cues
- Collapse repeated punctuation and whitespace
- Lowercase and trim

Quality filters:
- Remove placeholders/noise (`[deleted]`, `[removed]`, empty strings)
- Remove very short or non-alphabetic fragments
- Deduplicate exact rows by (`text_part`, `text`) within each topic

Why this matters:
- Increases SenticNet concept match reliability
- Reduces transformer tokenization noise
- Prevents spam/repeated comments from dominating predictions
- Preserves interpretability by keeping title/body/comment channels separate

In [19]:
import html

def preprocess(text: str) -> str:
    text = str(text)

    # Decode HTML entities and normalize unicode quotes/apostrophes
    text = html.unescape(text)
    text = text.replace("’", "'").replace("`", "'").replace("“", '"').replace("”", '"')

    # Remove URLs and Reddit user/subreddit mentions
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"/?[ur]/\w+", " ", text)

    # Remove markdown formatting and quote prefixes
    text = re.sub(r"\*+([^*]+)\*+", r"\1", text)
    text = re.sub(r"^>+", " ", text, flags=re.MULTILINE)

    # Keep basic punctuation useful for sentiment
    text = re.sub(r"[^\w\s!?.,'\-]", " ", text)

    # Collapse repeated punctuation and whitespace
    text = re.sub(r"([!?.,])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text


def is_valid_text(text: str) -> bool:
    """Filter noise, placeholders, and very short fragments."""
    if text is None:
        return False
    raw = str(text).strip()
    if raw == "":
        return False

    low = raw.lower()
    blocked = {
        "[deleted]", "[removed]", "deleted", "removed", "n/a", "na", "none"
    }
    if low in blocked:
        return False

    cleaned = preprocess(raw)
    if len(cleaned) < 3:
        return False
    # Require at least one alphabetic token to avoid pure symbols/numbers noise
    if re.search(r"[a-zA-Z]", cleaned) is None:
        return False

    return True

# Load your labelled evaluation file (must contain: text, gt_subjectivity, gt_polarity)
df_eval = pd.read_excel("eval.xlsx")
df_eval["text_clean"] = df_eval["text"].apply(preprocess)

print(f"Loaded eval rows: {len(df_eval)}")
print(df_eval[["text", "text_clean", "gt_subjectivity", "gt_polarity"]].head(3))

Loaded eval rows: 1000
                                                                                                                      text  \
0                                                              @xnausikaax oh no! where did u order from? that's horrible    
1  A great hard training weekend is over.  a couple days of rest and lets do it again!  Lots of computer time to put in...   
2                                                                 Right, off to work  Only 5 hours to go until I'm free xD   

                                                                                                               text_clean  \
0                                                               xnausikaax oh no! where did u order from? that's horrible   
1  a great hard training weekend is over. a couple days of rest and lets do it again! lots of computer time to put in now   
2                                                                 right, off to work only 5 hours

## Load Crawled Reddit Data

This section loads your crawled Reddit datasets from all 3 topics and creates a unified dataframe for large-scale analysis and random accuracy checks.

In [20]:
DATA_DIR = Path("../redditscrapper/data")
TOPICS = ["cryptocurrency", "Donald Trump", "Python programming"]

def collect_comments(comments, topic, post_id, bucket):
    for c in comments or []:
        author = c.get("author", "")
        body = c.get("body", "")

        if author not in ("[deleted]", "AutoModerator", "", None) and is_valid_text(body):
            bucket.append({
                "topic": topic,
                "source": "comment",
                "text_part": "comment_body",
                "post_id": post_id,
                "text": body,
            })

        collect_comments(c.get("replies", []), topic, post_id, bucket)

# Build one dataframe per file/topic with explicit fields:
# post_title, post_body, comment_body
topic_dfs = {}
for topic in TOPICS:
    rows = []
    file_path = DATA_DIR / topic / "enriched_results.json"
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    for post in payload.get("posts", []):
        p_author = post.get("author", "")
        post_id = post.get("id", "")

        title = post.get("title", "") or ""
        body = post.get("body", "") or ""

        if p_author not in ("[deleted]", "AutoModerator", "", None):
            if is_valid_text(title):
                rows.append({
                    "topic": topic,
                    "source": "post",
                    "text_part": "post_title",
                    "post_id": post_id,
                    "text": title,
                })
            if is_valid_text(body):
                rows.append({
                    "topic": topic,
                    "source": "post",
                    "text_part": "post_body",
                    "post_id": post_id,
                    "text": body,
                })

        collect_comments(post.get("comments", []), topic, post_id, rows)

    df_topic = pd.DataFrame(rows)
    # Remove exact duplicate text rows within each topic to reduce spam bias
    if len(df_topic) > 0:
        df_topic = df_topic.drop_duplicates(subset=["text_part", "text"]).reset_index(drop=True)
        df_topic["text_clean"] = df_topic["text"].apply(preprocess)

    topic_dfs[topic] = df_topic

# Combined view across all enriched_results files
df_all = pd.concat(topic_dfs.values(), ignore_index=True)

print("Per-file dataset sizes (after cleaning + dedup):")
for topic, dft in topic_dfs.items():
    print(f"\n- {topic}: {len(dft)} rows")
    if len(dft) > 0:
        print(dft["text_part"].value_counts().to_string())

print(f"\nCombined rows across all files: {len(df_all)}")

Per-file dataset sizes (after cleaning + dedup):

- cryptocurrency: 15237 rows
text_part
comment_body    15123
post_title         95
post_body          19

- Donald Trump: 17870 rows
text_part
comment_body    17763
post_title         98
post_body           9

- Python programming: 12217 rows
text_part
comment_body    12081
post_title         99
post_body          37

Combined rows across all files: 45324


## Classification Functions

We compare 3 approaches:
- SenticNet-only
- HuggingFace-only
- Hybrid (SenticNet subjectivity + HuggingFace polarity)

In [9]:
def senticnet_classify(text: str):
    scores = []
    for w in text.split():
        try:
            scores.append(float(sn.polarity_value(w)))
        except KeyError:
            pass

    if len(scores) == 0:
        return 0, None  # objective

    pol = 1 if float(np.mean(scores)) > 0 else 0
    return 1, pol


def hf_classify(text: str):
    label = hf_sentiment(text)[0]["label"]
    if label == "NEU":
        return 0, None
    return 1, (1 if label == "POS" else 0)


def hybrid_classify(text: str):
    subj, sn_pol = senticnet_classify(text)
    if subj == 0:
        return 0, None

    label = hf_sentiment(text)[0]["label"]
    if label == "NEU":
        return 1, (sn_pol if sn_pol is not None else 0)
    return 1, (1 if label == "POS" else 0)

print("All classifiers ready.")

All classifiers ready.


In [11]:
def macro_prf(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return p, r, f

summary_rows = []
gt_subj = df_eval["gt_subjectivity"].astype(int).tolist()

for name, pred_subj, t in [
    ("SenticNet-only", sn_subj, sn_t),
    ("HuggingFace-only", hf_subj, hf_t),
    ("Hybrid", hy_subj, hy_t),
]:
    p, r, f = macro_prf(gt_subj, pred_subj)
    summary_rows.append({
        "Approach": name,
        "Macro Precision": round(p, 4),
        "Macro Recall": round(r, 4),
        "Macro F1": round(f, 4),
        "Throughput(rec/s)": round(len(df_eval)/max(t, 1e-9), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

        Approach  Macro Precision  Macro Recall  Macro F1  Throughput(rec/s)
  SenticNet-only              0.5         0.400    0.4444           34561.70
HuggingFace-only              0.5         0.347    0.4097              39.07
          Hybrid              0.5         0.400    0.4444              71.69


In [10]:
def evaluate_approach(name, classify_fn, df):
    pred_subj, pred_pol = [], []

    t0 = time.time()
    for _, row in df.iterrows():
        s, p = classify_fn(row["text_clean"])
        pred_subj.append(s)
        pred_pol.append(p)
    elapsed = time.time() - t0

    print(f"\n{'='*70}")
    print(f"Approach: {name}")
    print(f"Rows: {len(df)} | Time: {elapsed:.2f}s | Throughput: {len(df)/max(elapsed, 1e-9):.2f} rec/s")

    print("\nSubtask 1: Subjectivity Detection")
    print(classification_report(
        df["gt_subjectivity"],
        pred_subj,
        target_names=["Objective", "Subjective"],
        zero_division=0,
    ))

    mask = df["gt_subjectivity"] == 1
    gt_pol = df.loc[mask, "gt_polarity"].fillna(0).astype(int).tolist()
    pred_pol_subj = [pred_pol[i] if pred_pol[i] is not None else 0 for i in df.index[mask]]

    print("Subtask 2: Polarity Detection (subjective only)")
    print(classification_report(
        gt_pol,
        pred_pol_subj,
        target_names=["Negative", "Positive"],
        zero_division=0,
    ))

    return pred_subj, pred_pol, elapsed

sn_subj, sn_pol_pred, sn_t = evaluate_approach("SenticNet-only", senticnet_classify, df_eval)
hf_subj, hf_pol_pred, hf_t = evaluate_approach("HuggingFace-only", hf_classify, df_eval)
hy_subj, hy_pol_pred, hy_t = evaluate_approach("Hybrid", hybrid_classify, df_eval)


Approach: SenticNet-only
Rows: 1000 | Time: 0.03s | Throughput: 34561.70 rec/s

Subtask 1: Subjectivity Detection
              precision    recall  f1-score   support

   Objective       0.00      0.00      0.00         0
  Subjective       1.00      0.80      0.89      1000

    accuracy                           0.80      1000
   macro avg       0.50      0.40      0.44      1000
weighted avg       1.00      0.80      0.89      1000

Subtask 2: Polarity Detection (subjective only)
              precision    recall  f1-score   support

    Negative       0.63      0.52      0.57       500
    Positive       0.59      0.70      0.64       500

    accuracy                           0.61      1000
   macro avg       0.61      0.61      0.61      1000
weighted avg       0.61      0.61      0.61      1000


Approach: HuggingFace-only
Rows: 1000 | Time: 25.59s | Throughput: 39.07 rec/s

Subtask 1: Subjectivity Detection
              precision    recall  f1-score   support

   Objective 

## Evaluation on Labelled Dataset

This section reports precision, recall, F1-score, and throughput for each approach.

In [12]:
# Performance and scalability summary
perf = pd.DataFrame([
    {"Approach": "SenticNet-only", "Rows": len(df_eval), "Seconds": round(sn_t, 3), "Rec/s": round(len(df_eval)/max(sn_t,1e-9), 2)},
    {"Approach": "HuggingFace-only", "Rows": len(df_eval), "Seconds": round(hf_t, 3), "Rec/s": round(len(df_eval)/max(hf_t,1e-9), 2)},
    {"Approach": "Hybrid", "Rows": len(df_eval), "Seconds": round(hy_t, 3), "Rec/s": round(len(df_eval)/max(hy_t,1e-9), 2)},
])

print("Performance comparison:")
print(perf.to_string(index=False))

print("\nScalability notes:")
print("- SenticNet-only is fastest and scales linearly with text volume.")
print("- HuggingFace and Hybrid are slower on CPU because transformer inference dominates runtime.")
print("- Throughput can improve significantly with batching + GPU.")

Performance comparison:
        Approach  Rows  Seconds    Rec/s
  SenticNet-only  1000    0.029 34561.70
HuggingFace-only  1000   25.592    39.07
          Hybrid  1000   13.948    71.69

Scalability notes:
- SenticNet-only is fastest and scales linearly with text volume.
- HuggingFace and Hybrid are slower on CPU because transformer inference dominates runtime.
- Throughput can improve significantly with batching + GPU.


In [21]:
SAMPLE_SIZE = 200
SPOTCHECK_SIZE = 2

def run_random_test_by_topic(topic, dft, classifier_fn=hybrid_classify):
    sample_df = dft.sample(min(SAMPLE_SIZE, len(dft)), random_state=42).copy()

    t0 = time.time()
    preds = sample_df["text_clean"].apply(classifier_fn)
    elapsed = time.time() - t0

    sample_df["pred_subjectivity"] = [p[0] for p in preds]
    sample_df["pred_polarity"] = [p[1] for p in preds]

    subj_counts = sample_df["pred_subjectivity"].map({0: "Objective", 1: "Subjective"}).value_counts().to_dict()
    pol_counts = sample_df["pred_polarity"].map({0: "Negative", 1: "Positive", None: "N/A"}).value_counts().to_dict()

    # Tiny spot-check output to keep notebook output manageable
    spot = sample_df.sample(min(SPOTCHECK_SIZE, len(sample_df)), random_state=99)[
        ["text", "pred_subjectivity", "pred_polarity"]
    ].copy()
    spot["pred_subjectivity"] = spot["pred_subjectivity"].map({0: "Objective", 1: "Subjective"})
    spot["pred_polarity"] = spot["pred_polarity"].map({0: "Negative", 1: "Positive", None: "N/A"})
    spot["text"] = spot["text"].astype(str).str.slice(0, 80)

    print(f"\n{topic} spot-check:")
    print(spot.to_string(index=False))

    return {
        "Topic": topic,
        "Rows Sampled": len(sample_df),
        "Hybrid rec/s": round(len(sample_df)/max(elapsed, 1e-9), 2),
        "Subjective": int(subj_counts.get("Subjective", 0)),
        "Objective": int(subj_counts.get("Objective", 0)),
        "Positive": int(pol_counts.get("Positive", 0)),
        "Negative": int(pol_counts.get("Negative", 0)),
        "N/A": int(pol_counts.get("N/A", 0)),
    }

random_test_rows = []
for topic, dft in topic_dfs.items():
    random_test_rows.append(run_random_test_by_topic(topic, dft))

random_test_summary = pd.DataFrame(random_test_rows)
print("\nRandom-test summary by data file:")
print(random_test_summary.to_string(index=False))


cryptocurrency spot-check:
                                                                            text pred_subjectivity pred_polarity
Guma to their amateur team, huanfeng academy and Ruler LCS, and still not even m        Subjective      Positive
         Looks good to me chief. I’m currently 212% YTD on NANO, while BTC is 5%        Subjective      Positive

Donald Trump spot-check:
                                                                            text pred_subjectivity pred_polarity
Sadly, he's already performed there to "honor" Led Zeppelin, delivering a shamef        Subjective      Negative
                                Get a third party to investigate and audit Patel        Subjective      Negative

Python programming spot-check:
                                                                            text pred_subjectivity pred_polarity
If u have a barebones kernel and C standart lib ig u can run python on ur own os        Subjective      Positive
          

## Full Classification on All Enriched Records (Per File)

After sample testing, this section performs **full classification** on every record in each `enriched_results.json` file.

Record definition used here:
- 1 record per post (using `title + body`)
- 1 record per comment (using comment `body`)

This matches metadata counts such as `Total_records = total_posts + total_comments`.

Output:
- Ranked per-record classification with confidence score
- One CSV per topic and one combined CSV

In [23]:
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_all_records_from_enriched(topic: str, payload: dict):
    rows = []

    def walk_comments(comments, post_id):
        for idx, c in enumerate(comments or []):
            body = str(c.get("body", "") or "")
            rows.append({
                "topic": topic,
                "record_type": "comment",
                "text_part": "comment_body",
                "post_id": post_id,
                "record_id": f"{post_id}:c:{len(rows)}",
                "text_raw": body,
            })
            walk_comments(c.get("replies", []), post_id)

    for post in payload.get("posts", []):
        post_id = str(post.get("id", ""))
        title = str(post.get("title", "") or "")
        body = str(post.get("body", "") or "")
        post_text = (title + " " + body).strip()

        rows.append({
            "topic": topic,
            "record_type": "post",
            "text_part": "post_title_body",
            "post_id": post_id,
            "record_id": f"{post_id}:p",
            "text_raw": post_text,
        })

        walk_comments(post.get("comments", []), post_id)

    return pd.DataFrame(rows)


def classify_full_topic(df_records: pd.DataFrame, batch_size: int = 32):
    dfr = df_records.copy()
    dfr["text_clean"] = dfr["text_raw"].apply(preprocess)

    # Handle empty/placeholder records without calling models
    invalid_mask = ~dfr["text_raw"].astype(str).apply(is_valid_text)
    dfr["subjectivity"] = 0
    dfr["polarity"] = None
    dfr["hf_label"] = "NEU"
    dfr["hf_score"] = 0.0

    # SenticNet subjectivity + fallback polarity
    sn_subj = []
    sn_pol = []
    for txt in dfr.loc[~invalid_mask, "text_clean"]:
        s, p = senticnet_classify(txt)
        sn_subj.append(s)
        sn_pol.append(p)

    valid_idx = dfr.index[~invalid_mask]
    dfr.loc[valid_idx, "subjectivity"] = sn_subj
    dfr.loc[valid_idx, "polarity"] = sn_pol

    # HF polarity for subjective rows only (hybrid)
    subj_idx = dfr.index[(~invalid_mask) & (dfr["subjectivity"] == 1)]
    if len(subj_idx) > 0:
        texts = dfr.loc[subj_idx, "text_clean"].tolist()
        hf_out = hf_sentiment(texts, batch_size=batch_size)

        hf_labels = [o["label"] for o in hf_out]
        hf_scores = [float(o["score"]) for o in hf_out]
        dfr.loc[subj_idx, "hf_label"] = hf_labels
        dfr.loc[subj_idx, "hf_score"] = hf_scores

        # Hybrid polarity decision
        for i in subj_idx:
            label = dfr.at[i, "hf_label"]
            if label == "POS":
                dfr.at[i, "polarity"] = 1
            elif label == "NEG":
                dfr.at[i, "polarity"] = 0
            else:
                # HF neutral on subjective text -> keep SenticNet fallback polarity
                if dfr.at[i, "polarity"] is None:
                    dfr.at[i, "polarity"] = 0

    # Final class labels
    def final_label(row):
        if int(row["subjectivity"]) == 0:
            return "Objective"
        return "Positive" if int(row["polarity"]) == 1 else "Negative"

    dfr["final_class"] = dfr.apply(final_label, axis=1)

    # Ranking score: signed confidence (positive > 0, negative < 0, objective = 0)
    def signed_score(row):
        if row["final_class"] == "Objective":
            return 0.0
        s = float(row["hf_score"])
        return s if row["final_class"] == "Positive" else -s

    dfr["rank_score"] = dfr.apply(signed_score, axis=1)
    dfr["rank_abs_conf"] = dfr["rank_score"].abs()
    dfr = dfr.sort_values(["rank_abs_conf", "rank_score"], ascending=[False, False]).reset_index(drop=True)
    dfr["rank"] = np.arange(1, len(dfr) + 1)

    return dfr

# Run full classification for each enriched file separately
full_results = {}
full_summary_rows = []

for topic in TOPICS:
    file_path = DATA_DIR / topic / "enriched_results.json"
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df_records = extract_all_records_from_enriched(topic, payload)
    total_meta = payload.get("metadata", {}).get("Total_records", None)

    t0 = time.time()
    df_classified = classify_full_topic(df_records, batch_size=32)
    elapsed = time.time() - t0

    # Save per-topic output
    safe_topic = topic.replace(" ", "_")
    out_path = OUTPUT_DIR / f"full_classification_{safe_topic}.csv"
    df_classified.to_csv(out_path, index=False)

    full_results[topic] = df_classified
    full_summary_rows.append({
        "topic": topic,
        "metadata_Total_records": total_meta,
        "classified_rows": len(df_classified),
        "rec_per_sec": round(len(df_classified) / max(elapsed, 1e-9), 2),
        "Positive": int((df_classified["final_class"] == "Positive").sum()),
        "Negative": int((df_classified["final_class"] == "Negative").sum()),
        "Objective": int((df_classified["final_class"] == "Objective").sum()),
        "output_file": str(out_path),
    })

# Save combined output
all_full = pd.concat(full_results.values(), ignore_index=True)
all_out_path = OUTPUT_DIR / "full_classification_all_topics.csv"
all_full.to_csv(all_out_path, index=False)

full_summary = pd.DataFrame(full_summary_rows)
print("Full-classification summary per data file:")
print(full_summary.to_string(index=False))
print(f"\nCombined output saved to: {all_out_path}")

# Quick check requested by user: Python programming should match metadata Total_records
pp = full_summary[full_summary["topic"] == "Python programming"].iloc[0]
print("\nPython programming count check:")
print(f"metadata Total_records = {pp['metadata_Total_records']}, classified_rows = {pp['classified_rows']}")

Full-classification summary per data file:
             topic  metadata_Total_records  classified_rows  rec_per_sec  Positive  Negative  Objective                                        output_file
    cryptocurrency                     NaN            18650        37.32      6841      7841       3968     outputs/full_classification_cryptocurrency.csv
      Donald Trump                 18962.0            18962        35.29      4378     11179       3405       outputs/full_classification_Donald_Trump.csv
Python programming                 14715.0            14715        31.74      7603      4012       3100 outputs/full_classification_Python_programming.csv

Combined output saved to: outputs/full_classification_all_topics.csv

Python programming count check:
metadata Total_records = 14715.0, classified_rows = 14715


## Random Accuracy Test on Remaining Data

This tests the Hybrid classifier on random records from the crawled Reddit corpus and prints a sample for manual verification.

In [22]:
from sklearn.metrics import cohen_kappa_score

print("Question 4 concise answer summary")
print("="*70)

# 1) Approach motivation + 2) preprocessing
print("1) Approach: Hybrid (SenticNet + HuggingFace transformer) to balance interpretability and SOTA accuracy.")
print("2) Preprocessing: URL/user/subreddit/markdown cleanup + normalization to improve KB and transformer robustness.")

# 3) Labelled dataset + IAA
auto_ann_cols = [c for c in df_eval.columns if str(c).lower().startswith("ann")]
print(f"3) Evaluation dataset size: {len(df_eval)} labelled rows.")
if len(auto_ann_cols) >= 2:
    print(f"   Annotator columns detected: {auto_ann_cols}")
    for i in range(len(auto_ann_cols)):
        for j in range(i+1, len(auto_ann_cols)):
            c1, c2 = auto_ann_cols[i], auto_ann_cols[j]
            m = df_eval[[c1, c2]].dropna()
            if len(m) > 0:
                agree = (m[c1] == m[c2]).mean()
                kappa = cohen_kappa_score(m[c1], m[c2])
                print(f"   {c1} vs {c2}: agreement={agree*100:.2f}% | kappa={kappa:.3f}")
else:
    print("   IAA columns not detected in eval.xlsx. Add columns like ann1, ann2 (and ann3) to compute >=80% agreement.")

# 4) Metrics
print("4) Evaluation metrics (from labelled set):")
print(summary_df.to_string(index=False))

# 5) Random test by topic file
print("5) Random test by data file:")
print(random_test_summary.to_string(index=False))

# 6) Performance/scalability
print("6) Performance (records/sec on labelled set):")
print(perf.to_string(index=False))
print("   Scalability: SenticNet fastest; Hybrid/HF limited by transformer inference on CPU; batching/GPU recommended.")

Question 4 concise answer summary
1) Approach: Hybrid (SenticNet + HuggingFace transformer) to balance interpretability and SOTA accuracy.
2) Preprocessing: URL/user/subreddit/markdown cleanup + normalization to improve KB and transformer robustness.
3) Evaluation dataset size: 1000 labelled rows.
   IAA columns not detected in eval.xlsx. Add columns like ann1, ann2 (and ann3) to compute >=80% agreement.
4) Evaluation metrics (from labelled set):
        Approach  Macro Precision  Macro Recall  Macro F1  Throughput(rec/s)
  SenticNet-only              0.5         0.400    0.4444           34561.70
HuggingFace-only              0.5         0.347    0.4097              39.07
          Hybrid              0.5         0.400    0.4444              71.69
5) Random test by data file:
             Topic  Rows Sampled  Hybrid rec/s  Subjective  Objective  Positive  Negative  N/A
    cryptocurrency           200         47.89         162         38        78        84   38
      Donald Trump    

## How The Pipeline Runs 

### End-to-end flow
1. Load each topic file from `redditscrapper/data/<topic>/enriched_results.json`.
2. Build records as:
   - one record per post using `title + body`
   - one record per comment using comment `body` (recursive replies included)
3. Clean text with microtext normalization.
4. Run **subjectivity detection** using SenticNet.
5. Run **polarity detection** (only for subjective records) using HuggingFace sentiment model.
6. Produce final class and ranking score.

### What "subjective" and "objective" mean in this notebook
- **Objective**: informational/factual text with no strong opinion signal under the subjectivity gate.
- **Subjective**: text that expresses sentiment/opinion and should be passed to polarity classification.

Operationally in this notebook:
- If no reliable sentiment evidence is found in the subjectivity stage, the item is marked **Objective**.
- If subjective, it is classified as **Positive** or **Negative** in polarity stage.

### Why this method was chosen
- Transformer models are state-of-the-art for polarity prediction, but can over-assign sentiment on factual text.
- A subjectivity gate reduces false sentiment assignments on objective posts/comments.
- Hybrid design provides better interpretability and practical control for social media text.

## Question 4 Detailed Answers

### 1) Motivate the classification approach in relation with state of the art
- We adopt a **hybrid** pipeline: knowledge-based SenticNet + transformer-based HuggingFace model.
- State-of-the-art NLP sentiment systems are transformer-dominant; we use that strength for polarity.
- We keep a knowledge-based gate to explicitly model subjectivity before polarity, improving robustness on mixed factual/opinion Reddit content.

### 2) Discuss preprocessing and why
We apply microtext normalization because Reddit text is noisy:
- decode HTML entities
- remove URLs, `u/` and `r/` mentions
- strip markdown emphasis/quotes
- normalize punctuation and whitespace
- lowercase text
- remove placeholders (`[deleted]`, `[removed]`) and near-empty noise
- deduplicate repeated rows by text channel

Why needed:
- improves SenticNet concept matching
- reduces tokenization noise in transformer inference
- reduces spam/duplicate bias in class distributions

### 3) Build evaluation dataset (>=1000) and inter-annotator agreement >=80%
- The notebook uses `eval.xlsx` with 1000 manually labeled rows.
- Annotator-agreement logic is included and computes pairwise agreement and Cohen's kappa when annotator columns are present.
- If annotator columns are missing, the notebook explicitly flags this and recommends adding `ann1`, `ann2`, `ann3`.

### 4) Provide precision, recall, F-measure
- The notebook reports classification metrics for:
  - subjectivity task
  - polarity task (on subjective rows)
- It compares SenticNet-only, HuggingFace-only, and Hybrid.

### 5) Perform random accuracy test and discuss results
- The notebook runs random tests separately for each topic file.
- It reports class distributions and spot-check samples.
- Topic patterns are discussed from outputs (for example, political topic tends to more negative predictions).

### 6) Discuss performance and scalability
- Records/second is measured on labeled evaluation and full-run classification.
- SenticNet is fastest.
- Transformer inference dominates runtime on CPU.
- Batching and GPU are the main scaling levers.

## Detailed Method Internals: SenticNet -> HuggingFace

### A) SenticNet stage (how subjectivity is decided)

In this notebook, SenticNet is used as a **knowledge-based scoring function** over cleaned tokens.

For a cleaned text $x$:
1. Tokenize into words $w_1, w_2, ..., w_n$.
2. For each token $w_i$, query Sentic polarity value $p(w_i)$.
3. Keep only matched tokens (tokens that exist in SenticNet):
   $$M = \{w_i \mid p(w_i)\ \text{exists}\}$$

Then compute:
- Match count: $m = |M|$
- Mean Sentic polarity:
  $$\bar{p} = \frac{1}{m}\sum_{w\in M} p(w)\quad\text{(if }m>0\text{)}$$

Decision rules in the current code:
- **Subjectivity**:
  $$\text{subjective}=\begin{cases}1,& m>0\\0,& m=0\end{cases}$$
  - `1` = Subjective
  - `0` = Objective
- **Fallback polarity** (only if subjective):
  $$\text{polarity}=\begin{cases}1,& \bar{p}>0\\0,& \bar{p}\le 0\end{cases}$$
  - `1` = Positive
  - `0` = Negative

Interpretation:
- Objective here means: "no Sentic sentiment evidence found after preprocessing".
- Subjective means: "at least one sentiment-bearing concept/token found".

### B) HuggingFace stage (how polarity is decided after Sentic gate)

Only records marked subjective by SenticNet are passed to HuggingFace.

For each subjective text, HuggingFace returns:
- `label` in `{POS, NEG, NEU}`
- `score` in $[0,1]$ (model confidence for top label)

Hybrid polarity mapping:
- If `label == POS` -> Positive (`polarity = 1`)
- If `label == NEG` -> Negative (`polarity = 0`)
- If `label == NEU` -> use Sentic fallback polarity (from $\bar{p}$ rule)

So final class is:
- Objective if `subjective = 0`
- Positive or Negative if `subjective = 1`

### C) Ranking output (full classification section)

For full-record classification, confidence ranking is computed as signed score:
- Positive: `rank_score = +hf_score`
- Negative: `rank_score = -hf_score`
- Objective: `rank_score = 0`

Rows are sorted by absolute confidence `|rank_score|` (strongest confidence first), then assigned rank 1..N.

### D) Why this two-stage design is used

- SenticNet provides transparent, knowledge-based subjectivity gating.
- HuggingFace provides stronger contextual polarity prediction (state-of-the-art transformer behavior).
- The hybrid avoids forcing sentiment labels on clearly objective/noisy text while still using deep learning where it helps most.

## Pipeline Flowchart

```mermaid
flowchart TD
    A[Load enriched_results.json per topic] --> B[Build records]
    B --> B1[Post record: title + body]
    B --> B2[Comment records: recursive comment body]

    B1 --> C[Preprocess text]
    B2 --> C

    C --> D{Valid text?}
    D -- No --> O[Objective\nsubjectivity=0\nfinal_class=Objective]
    D -- Yes --> E[SenticNet token lookup]

    E --> F{Any Sentic match?\n m > 0}
    F -- No --> O
    F -- Yes --> G[Subjective\nsubjectivity=1]

    G --> H[HuggingFace sentiment\nlabel in POS/NEG/NEU + score]
    H --> I{HF label}

    I -- POS --> P[polarity=1\nfinal_class=Positive]
    I -- NEG --> N[polarity=0\nfinal_class=Negative]
    I -- NEU --> J[Fallback to Sentic mean polarity]

    J --> K{mean Sentic polarity > 0?}
    K -- Yes --> P
    K -- No --> N

    O --> R[rank_score = 0]
    P --> R1[rank_score = +hf_score]
    N --> R2[rank_score = -hf_score]

    R --> S[Rank by abs(rank_score)]
    R1 --> S
    R2 --> S

    S --> T[Export CSV per topic + combined CSV]
```

In [25]:
# Quality audit + improvement recommendations for Question 4
from sklearn.metrics import cohen_kappa_score

print("Question 4 quality audit")
print("="*80)

# A) Dataset coverage checks
print(f"Labelled dataset rows: {len(df_eval)}")
if len(df_eval) < 1000:
    print("[WARN] eval.xlsx has fewer than 1000 rows.")
else:
    print("[OK] eval.xlsx has at least 1000 rows.")

if "gt_subjectivity" in df_eval.columns:
    subj_dist = df_eval["gt_subjectivity"].value_counts(dropna=False).to_dict()
    print(f"Subjectivity distribution: {subj_dist}")
    has_obj = 0 in subj_dist
    has_subj = 1 in subj_dist
    if not (has_obj and has_subj):
        print("[WARN] Subjectivity labels do not contain both classes (0 and 1).")
        print("       Add objective examples so Subjectivity metrics are meaningful.")
    else:
        print("[OK] Subjectivity has both objective and subjective classes.")

if "gt_polarity" in df_eval.columns:
    pol_dist = df_eval["gt_polarity"].value_counts(dropna=False).to_dict()
    print(f"Polarity distribution: {pol_dist}")

# B) Inter-annotator agreement checks
ann_cols = [c for c in df_eval.columns if str(c).lower().startswith("ann")]
if len(ann_cols) >= 2:
    print(f"Annotator columns detected: {ann_cols}")
    pair_rows = []
    for i in range(len(ann_cols)):
        for j in range(i + 1, len(ann_cols)):
            c1, c2 = ann_cols[i], ann_cols[j]
            m = df_eval[[c1, c2]].dropna()
            if len(m) == 0:
                continue
            agree = float((m[c1] == m[c2]).mean())
            kappa = float(cohen_kappa_score(m[c1], m[c2]))
            pair_rows.append((c1, c2, len(m), agree, kappa))

    if pair_rows:
        print("Pairwise IAA:")
        for c1, c2, n, agree, kappa in pair_rows:
            status = "OK" if agree >= 0.80 else "WARN"
            print(f"[{status}] {c1} vs {c2} | n={n} | agreement={agree*100:.2f}% | kappa={kappa:.3f}")
    else:
        print("[WARN] Annotator columns exist but no overlapping non-null rows.")
else:
    print("[WARN] No annotator columns found for IAA check (expected ann1/ann2/ann3).")

# C) Full-run record count check against metadata
if 'full_summary' in globals():
    print("\nFull-run metadata vs classified rows:")
    print(full_summary[["topic", "metadata_Total_records", "classified_rows"]].to_string(index=False))

# D) Improvement suggestions
print("\nRecommended improvements:")
print("1) Add balanced objective records in eval.xlsx (not only subjective rows).")
print("2) Add annotator columns (ann1, ann2, ann3) and target >=80% agreement.")
print("3) Keep separate reports for post_title_body vs comment_body to analyze channel effects.")
print("4) For better speed, run HuggingFace on GPU and increase batch size.")

Question 4 quality audit
Labelled dataset rows: 1000
[OK] eval.xlsx has at least 1000 rows.
Subjectivity distribution: {1: 1000}
[WARN] Subjectivity labels do not contain both classes (0 and 1).
       Add objective examples so Subjectivity metrics are meaningful.
Polarity distribution: {0: 500, 1: 500}
[WARN] No annotator columns found for IAA check (expected ann1/ann2/ann3).

Full-run metadata vs classified rows:
             topic  metadata_Total_records  classified_rows
    cryptocurrency                     NaN            18650
      Donald Trump                 18962.0            18962
Python programming                 14715.0            14715

Recommended improvements:
1) Add balanced objective records in eval.xlsx (not only subjective rows).
2) Add annotator columns (ann1, ann2, ann3) and target >=80% agreement.
3) Keep separate reports for post_title_body vs comment_body to analyze channel effects.
4) For better speed, run HuggingFace on GPU and increase batch size.
